# 05. CIES Experiment trên Dataset Phụ (ULB Credit Card Fraud)

## Mục tiêu
Notebook 04 đo `cies_score` (độ ổn định giải thích SHAP qua bootstrap) trên dataset chính (Sparkov).
Notebook này chạy LẠI đúng cơ chế CIES đó trên dataset phụ **ULB** (`mlg-ulb/creditcardfraud`)
để kiểm tra: với cùng model × kỹ thuật imbalance, `cies_score` có nhất quán giữa 2 dataset không,
hay độ ổn định giải thích phụ thuộc nhiều vào đặc điểm riêng của từng dataset.

## Khác biệt so với Sparkov (notebook 04)
- **Không cần bước encoding categorical**: ULB đã là toàn bộ feature số
  (`V1`..`V28` — PCA components, `Amount`, `Time`). Gọi `run_cies_experiment_isolated()`
  với `onehot_cols=[]`, `target_encode_cols=[]` khiến `encode_train`/`encode_test` bên trong
  trở thành no-op (pass-through) — không cần sửa gì ở `src/data/encoding.py` hay `src/explainability/cies.py`,
  vì cả 2 hàm vốn đã dataset-agnostic (chỉ lặp qua đúng những cột được truyền vào).
- **Target column**: `Class` (không phải `is_fraud`) — dùng `ULB_TARGET_COL` từ `src/config.py`.
- **`feature_level=False`**: giữ nguyên chủ đích thiết kế gốc trong `cies.py`
  ("chỉ dùng cho dataset chính, không cần cho dataset phụ") — feature V1..V28 là PCA component
  ẩn danh, không có ý nghĩa diễn giải trực tiếp như feature gốc của Sparkov.
- **Lưu kết quả riêng**: `results/cies_summary_results_ulb.json` (không ghi đè file của Sparkov).
- **Resilience**: mỗi tổ hợp (model × technique) được lưu NGAY sau khi chạy xong, và bỏ qua tổ hợp
  đã có sẵn trong file kết quả nếu chạy lại — tránh mất dữ liệu nếu 1 tổ hợp bị timeout/lỗi
  giữa chừng (bài học từ lần Optuna HPO bị timeout mất kết quả trên Kaggle).

In [ ]:
# 1. Setup & Imports
import sys
from pathlib import Path
sys.path.insert(0, '..')

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    PROCESSED_DATA_DIR, RESULTS_DIR, SEED, N_RUNS,
    ULB_TARGET_COL, MODEL_NAMES, IMBALANCE_TECHNIQUES,
)
from src.explainability.cies import run_cies_experiment_isolated, save_cies_results

print(f'SEED: {SEED}, N_RUNS config: {N_RUNS}')
print('Models:', MODEL_NAMES)
print('Imbalance techniques:', IMBALANCE_TECHNIQUES)

In [ ]:
# 2. Load ULB train/test (RAW — đã là feature số sẵn, không cần encode)
ulb_train = pd.read_parquet(PROCESSED_DATA_DIR / 'ulb_train.parquet')
ulb_test = pd.read_parquet(PROCESSED_DATA_DIR / 'ulb_test.parquet')

print(f'ULB train: {ulb_train.shape[0]:,} dòng x {ulb_train.shape[1]} cột')
print(f'ULB test:  {ulb_test.shape[0]:,} dòng x {ulb_test.shape[1]} cột')
print(f'Fraud rate (train): {ulb_train[ULB_TARGET_COL].mean():.4%}')

# Fixed eval set — GIỮ NGUYÊN qua toàn bộ N_RUNS (giống notebook 04)
eval_fraud = ulb_test[ulb_test[ULB_TARGET_COL] == 1].sample(
    n=min(50, int(ulb_test[ULB_TARGET_COL].sum())), random_state=SEED)
eval_legit = ulb_test[ulb_test[ULB_TARGET_COL] == 0].sample(
    n=min(200, int((ulb_test[ULB_TARGET_COL] == 0).sum())), random_state=SEED)
df_test_fixed_eval = pd.concat([eval_fraud, eval_legit]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print(f'\nFixed Eval Set: {len(df_test_fixed_eval)} mẫu (Fraud: {df_test_fixed_eval[ULB_TARGET_COL].sum()})')

# KHÔNG subsample — dùng NGUYÊN toàn bộ ULB train (227,845 dòng) cho bootstrap.
# Khác với Sparkov (1.48M dòng, phải subsample xuống 10,000 để chạy được cục
# bộ — xem notebook 04), ULB đủ nhỏ để train trực tiếp trên toàn bộ dữ liệu
# trong thời gian hợp lý (đã đo: RandomForest ~27s/fit, XGBoost <1s/fit trên
# full 227K dòng — tổng ~30-40 phút cho 9 tổ hợp x N_RUNS=20). Dùng full
# dataset loại bỏ hoàn toàn câu hỏi "subsample có công bằng với Sparkov
# không" — không còn điều chỉnh cỡ mẫu/tỷ lệ nhân tạo nào cả, đúng 100% dữ
# liệu gốc, giống hệt cách notebook 03 (benchmark) dùng full Sparkov train.
ulb_train_sub = ulb_train
print(f'Train cho bootstrap: TOÀN BỘ {len(ulb_train_sub):,} dòng (không subsample), '
      f'fraud: {int(ulb_train_sub[ULB_TARGET_COL].sum())}')

---
## 3. Chạy Thí Nghiệm CIES trên ULB
Chạy cho các tổ hợp đại diện trước (giống scope hiện tại của notebook 04) — có thể mở rộng
`ULB_SUBSET_MODELS`/`ULB_SUBSET_TECHNIQUES` thành đủ `MODEL_NAMES`/`IMBALANCE_TECHNIQUES`
khi chạy full scale trên Kaggle.

In [ ]:
# 3. Run CIES Experiment Runner — ULB
CURRENT_N_RUNS = N_RUNS
ULB_SUBSET_MODELS = ['logistic_regression', 'random_forest', 'xgboost']
ULB_SUBSET_TECHNIQUES = ['class_weighting', 'smote', 'smote_enn']

out_file = RESULTS_DIR / 'cies_summary_results_ulb.json'

cies_ulb_results = []
if out_file.exists():
    with open(out_file, 'r', encoding='utf-8') as f:
        cies_ulb_results = json.load(f)
    print(f'Đã tìm thấy {len(cies_ulb_results)} tổ hợp từ lần chạy trước — sẽ bỏ qua, chỉ chạy tổ hợp còn thiếu.')

done_combos = {
    (r.get('model_name'), r.get('imbalance_technique'))
    for r in cies_ulb_results
}

for model_name in ULB_SUBSET_MODELS:
    for technique in ULB_SUBSET_TECHNIQUES:
        if (model_name, technique) in done_combos:
            print(f'⏭️  Bỏ qua (đã có): {model_name} x {technique}')
            continue

        print(f'\n{"="*40}')
        print(f'Running CIES (ULB): Model={model_name}, Technique={technique}')

        try:
            res = run_cies_experiment_isolated(
                model_name=model_name,
                imbalance_technique=technique,
                df_train=ulb_train_sub,
                df_test_fixed_eval=df_test_fixed_eval,
                target_col=ULB_TARGET_COL,
                onehot_cols=[],
                target_encode_cols=[],
                n_runs=CURRENT_N_RUNS,
                feature_level=False,
            )
            print(f'--> CIES Score: {res["cies_metrics"]["cies_score"]:.4f} '
                  f'(Spearman: {res["cies_metrics"]["mean_spearman"]:.4f})')
        except (TimeoutError, RuntimeError) as e:
            print(f'  ❌ Thất bại: {type(e).__name__}: {e}')
            res = {
                'model_name': model_name,
                'imbalance_technique': technique,
                'error': f'{type(e).__name__}: {e}',
            }

        cies_ulb_results.append(res)

        # Lưu NGAY sau mỗi tổ hợp — 1 tổ hợp lỗi/timeout không làm mất các
        # tổ hợp đã chạy xong trước đó (xem notebooks/kaggle_optuna_tuning.ipynb
        # và src/models/tune.py::tune_all_models cho cùng bài học).
        save_cies_results(cies_ulb_results, RESULTS_DIR, filename='cies_summary_results_ulb.json')

print(f'\n✅ Hoàn thành. Kết quả: {out_file}')

---
## 4. Bảng CIES — ULB

In [ ]:
# 4. Bảng tổng hợp CIES (ULB)
summary_rows = []
for r in cies_ulb_results:
    if 'cies_metrics' not in r:
        continue
    summary_rows.append({
        'model': r['model_name'],
        'technique': r['imbalance_technique'],
        'cies_score': r['cies_metrics']['cies_score'],
        'mean_rank_distance': r['cies_metrics']['mean_rank_distance'],
        'std_rank_distance': r['cies_metrics']['std_rank_distance'],
        'mean_spearman': r['cies_metrics']['mean_spearman'],
    })

df_cies_ulb = pd.DataFrame(summary_rows)
print('=== BẢNG CIES — ULB ===')
df_cies_ulb.sort_values('cies_score', ascending=False)

---
## 5. So sánh CIES Score: Sparkov vs. ULB
Ghép với `results/cies_summary_results.json` (Sparkov, notebook 04) trên các tổ hợp model ×
kỹ thuật CHUNG giữa 2 dataset, để xem `cies_score` có nhất quán (generalize) không.

In [ ]:
# 5. So sánh cies_score giữa Sparkov và ULB (cùng model x technique)
sparkov_path = RESULTS_DIR / 'cies_summary_results.json'

if sparkov_path.exists():
    with open(sparkov_path, 'r', encoding='utf-8') as f:
        cies_sparkov_raw = json.load(f)

    sparkov_rows = [
        {
            'model': r['model_name'],
            'technique': r['imbalance_technique'],
            'cies_score_sparkov': r['cies_metrics']['cies_score'],
        }
        for r in cies_sparkov_raw
        if 'cies_metrics' in r
    ]
    df_sparkov = pd.DataFrame(sparkov_rows)

    df_compare = pd.merge(
        df_cies_ulb.rename(columns={'cies_score': 'cies_score_ulb'})[['model', 'technique', 'cies_score_ulb']],
        df_sparkov,
        on=['model', 'technique'],
        how='inner',
    )
    df_compare['abs_diff'] = (df_compare['cies_score_ulb'] - df_compare['cies_score_sparkov']).abs()

    print(f'{len(df_compare)} tổ hợp chung giữa Sparkov và ULB:')
    display(df_compare.sort_values('abs_diff'))

    if len(df_compare) > 0:
        plt.figure(figsize=(8, 6))
        plt.scatter(df_compare['cies_score_sparkov'], df_compare['cies_score_ulb'])
        for _, row in df_compare.iterrows():
            plt.annotate(f"{row['model']}\n{row['technique']}", (row['cies_score_sparkov'], row['cies_score_ulb']), fontsize=8)
        lims = [0, 1]
        plt.plot(lims, lims, 'k--', alpha=0.4, label='y = x (nhất quán hoàn toàn)')
        plt.xlabel('CIES Score — Sparkov')
        plt.ylabel('CIES Score — ULB')
        plt.title('CIES Score: Sparkov vs. ULB (cùng model x kỹ thuật)')
        plt.legend()
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / 'cies_sparkov_vs_ulb.png', dpi=150)
        plt.show()
else:
    print(f'Chưa có {sparkov_path} — chạy notebooks/04_cies_experiment.ipynb trước để so sánh được.')